# Cleaning Data — Oasis Infobyte Data Analytics

## Task 3 · Level 1

**Objective:** Demonstrate professional data cleaning by transforming a deliberately messy dataset into a clean, analysis-ready dataset and documenting each decision.

**Tech Stack:** Python, pandas, numpy, Jupyter Notebook.


## 1. Load Dataset and Initial Data Quality Report

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("messy_customer_data.csv")

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nNull values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

print("\nNumeric summary:")
print(df[["Purchase_Amount", "Age"]].describe())

print("\nUnique values / formatting examples:")
print("Gender:", df["Gender"].dropna().unique())
print("City:", df["City"].dropna().unique())


### Observation

The initial report checks the required data-quality areas: null values, duplicate rows, data types and numeric ranges. The dataset contains intentionally messy formatting, missing values, duplicate rows and invalid age values.


## 2. Missing Data Handling

In [ ]:
# Missing-value decisions:
# Gender -> mode because it is categorical.
# City -> mode because it is categorical.
# Purchase_Date -> row deletion because a transaction without a date
# cannot be reliably placed in the time sequence.
# Purchase_Amount -> median because monetary data can be affected by outliers.
# Age -> median because it is numeric and robust to extreme values.

before_missing = df.isnull().sum()

df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])
df["City"] = df["City"].fillna(df["City"].mode()[0])
df["Purchase_Amount"] = df["Purchase_Amount"].fillna(df["Purchase_Amount"].median())
df["Age"] = df["Age"].fillna(df["Age"].median())

df = df.dropna(subset=["Purchase_Date"]).copy()

print("Missing values before:")
print(before_missing)
print("\nMissing values after initial handling:")
print(df.isnull().sum())


### Justification

- **Mode imputation** is used for categorical Gender and City.
- **Median imputation** is used for Purchase Amount because extreme values can make the mean less representative.
- **Median imputation** is used for Age for the same robustness reason.
- Rows missing Purchase Date are deleted because the date is important for time-based interpretation and cannot be safely guessed.


## 3. Duplicate Removal

In [ ]:
duplicates_before = df.duplicated().sum()
df = df.drop_duplicates().copy()
duplicates_after = df.duplicated().sum()

print("Duplicates removed:", duplicates_before)
print("Duplicates remaining:", duplicates_after)


## 4. Standardisation of Inconsistent Formatting

In [ ]:
df["Gender"] = (
    df["Gender"].astype(str).str.strip().str.lower()
    .replace({"m": "Male", "male": "Male", "f": "Female", "female": "Female"})
)

df["City"] = (
    df["City"].astype(str).str.strip().str.title()
)

df["Customer_ID"] = df["Customer_ID"].astype("string")
df["Customer_Name"] = df["Customer_Name"].astype("string")

print("Gender values:", df["Gender"].unique())
print("City values:", df["City"].unique())


### Observation

Inconsistent values such as `male`, `Male`, `M` and different capitalization of city names are standardized so the same category is represented consistently.


## 5. Data Type Correction

In [ ]:
df["Purchase_Date"] = pd.to_datetime(df["Purchase_Date"], errors="coerce")
df["Purchase_Amount"] = pd.to_numeric(df["Purchase_Amount"], errors="coerce")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

print(df.dtypes)


## 6. Outlier Detection Using IQR

In [ ]:
def iqr_report(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (data[column] < lower) | (data[column] > upper)
    return q1, q3, lower, upper, int(mask.sum())

for col in ["Purchase_Amount", "Age"]:
    q1, q3, lower, upper, count = iqr_report(df, col)
    print(f"{col}: Q1={q1:.2f}, Q3={q3:.2f}, Lower={lower:.2f}, Upper={upper:.2f}, Outliers={count}")


### Outlier Decision

For **Purchase Amount**, extreme values are retained because a genuinely expensive purchase can be a valid business transaction.

For **Age**, values outside the realistic range of 18–100 are treated as data errors. They are replaced with the median age rather than deleting the entire customer row.


In [ ]:
median_age = df.loc[df["Age"].between(18, 100), "Age"].median()
invalid_age_mask = ~df["Age"].between(18, 100)
invalid_age_count = int(invalid_age_mask.sum())

df.loc[invalid_age_mask, "Age"] = median_age

print("Invalid age values corrected:", invalid_age_count)
print("Age range after correction:", df["Age"].min(), "to", df["Age"].max())


## 7. Final Data Quality Check

In [ ]:
quality_report_after = pd.DataFrame({
    "Null_Count": df.isnull().sum(),
    "Data_Type": df.dtypes.astype(str)
})

print(quality_report_after)
print("\nDuplicate rows:", df.duplicated().sum())
print("Rows:", len(df))


## 8. Before vs After Summary

In [ ]:
before_summary = pd.DataFrame({
    "Metric": ["Row Count", "Null Cell Count", "Duplicate Rows", "Invalid Age Values"],
    "Before Cleaning": [
        510,
        int(pd.read_csv("messy_customer_data.csv").isnull().sum().sum()),
        int(pd.read_csv("messy_customer_data.csv").duplicated().sum()),
        int((pd.to_numeric(pd.read_csv("messy_customer_data.csv")["Age"], errors="coerce").dropna().between(18,100) == False).sum())
    ],
    "After Cleaning": [
        len(df),
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum()),
        int((~df["Age"].between(18,100)).sum())
    ]
})

before_summary


## 9. Save Cleaned Dataset

In [ ]:
cleaned_path = "cleaned_customer_data.csv"
df.to_csv(cleaned_path, index=False)

print("Cleaned dataset saved as:", cleaned_path)
print("Final shape:", df.shape)


## Conclusion

The deliberately messy customer dataset has been transformed into an analysis-ready dataset.

### Cleaning work completed
1. Created a data quality report.
2. Handled missing values using documented strategies.
3. Removed duplicate rows.
4. Standardized inconsistent categorical formatting.
5. Detected numeric outliers using the IQR method.
6. Corrected invalid age values.
7. Corrected data types.
8. Produced a before-vs-after summary.
9. Saved the cleaned dataset as a new CSV file.

**Task 3 — Cleaning Data completed.**
